# 🎯 [MOV01_M10] 전시 수송 병목 구간 식별 및 우회 수송 시뮬레이션 도구 통합 패키지 완성 (정답 솔루션)
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV01/mov01_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 3) |
| **미션 ID** | MOV01_M10 |
| **미션 주제** | 전시 수송 병목 구간 식별 및 우회 수송 시뮬레이션 도구 통합 패키지 완성 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수송 소요를 데이터로 예측해 군수 수송계획을 세우는 일은 구간별 통과 용량과 병목을 함께 보아야 정확해진다. 본 프로젝트는 수송 소요예측에서 예측 대상 정의, 시계열·피처 구성, 기준모델 비교, 설명가능성, 업무 임계값, 실적 대조를 익히도록 교육용 가상 노선망 자료를 코드로 만들어 구간별 수송 소요와 지체를 예측한다. 학습자는 병목 데이터 정의, 지체 전처리, BPR 기준 모델, 최대유량 기반 병목 식별과 피처 모델, 우회 소요·유류 산출과 앙상블 모델, 혼잡도 대시보드와 요인 설명, 작전군수관 승인(HITL)과 선형계획 최적화, 3개 구간 동시 차단 스트레스 시험, AAR 재보정과 ROI, 재학습 파이프라인 통합을 차례로 수행한다. 최종 목표는 구간별 소요·지체를 예측해 병목을 사전에 찾고 우회 물량 배분안을 만들되, 우회 명령의 하달 여부는 반드시 사람이 결정하는 수송 소요예측 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(작전군수처, 수송대대)에 즉시 도입 가능한 "완성형 국방 AI 수송 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 기술 명세 및 운영 매뉴얼** | MSR 통과 능력 데이터 명세서(`docs/mov01_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mov01_operations_guide.md`) | 미션 1, 10 |
| **`src/`** | **군수 데이터 정제 및 분석 모듈** | 통과 지체시간 계산 전처리 파이프라인(`src/data_preprocessing.py`), BPR 기반 1차 지체 예측 Baseline(`src/baseline_delay.py`) | 미션 2, 3 |
| **`notebooks/`** | **통계 검증 및 심층 EDA** | MSR 구간 지체 탐색적 데이터 분석 및 파레토 분석(`notebooks/mov01_eda_analysis.ipynb`) | 미션 3 |
| **`src/`** | **AI 예측 & 우회로 탐색 엔진** | Max-Flow 기반 병목 사전 식별 및 우회로 탐색(`src/network_flow.py`), 유류 소모 및 소요시간 산출(`src/route_cost_engine.py`) | 미션 4, 5 |
| **`src/`** | **작전 상황판 & 스트레스 모의** | Streamlit 기반 전술 COP 대시보드(`src/dashboard_app.py`), 3개 축선 봉쇄 스트레스 시험(`src/stress_test.py`) | 미션 6, 8 |
| **`pipeline/`** | **HITL 통제 & 지속적 학습** | 작전군수관 3단계 승인 파이프라인(`pipeline/hitl_workflow.py`), 챔피언/챌린저 모델 자동 재학습 | 미션 7, 10 |
| **`reports/`** | **검증 및 AAR 리포트** | 3개 축선 봉쇄 스트레스 검증 보고서(`reports/mov01_stress_test_report.md`), AAR 오차 분석 및 ROI 보고서(`reports/mov01_roi_analysis.md`) | 미션 8, 9 |
| **`root`** | **통합 배포 패키지** | 통합 실행 진입점(`main.py`), 국방 망분리 Dockerfile 패키지 | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 전시 수송 병목 구간 식별 및 우회 수송 시뮬레이션 도구 통합 패키지 완성
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전시 수송 병목 구간 식별 및 우회 수송 시뮬레이션 도구를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **데이터 갱신부터 정합성 검증, 드리프트 점검, 자동 재학습, 7일 소요 예측, 우회 배분, 승인대기 분리까지 한 번에 실행되는 통합 파이프라인을 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **신규 30일 데이터를 추가해 정합성을 검증하고, 기존 모델의 오차로 드리프트를 판단한 뒤 챌린저 모델을 학습해 개선이 3% 이상일 때만 승격 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/mov01_operations_guide.md 및 pipeline/mov01_retrain.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지속 가능한 운영 가능성 및 국방 군수 정보체계와의 연동 용이성

#### 미션 PASS 조건

- 정량 기준: 자동 재학습 파이프라인 정상 실행, 최종 문서 100% 완비


- 진행 기준: 최종 인도물 검수 및 프로젝트 완료 발표자료 완성


- 검증 방법: End-to-End 전체 파이프라인 자동화 실행 및 최종 검수


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""데이터 갱신, 정합성 검증, 드리프트 점검, 자동 재학습(챔피언/챌린저), 7일 수송 소요 예측, 병목·우회 배분, 승인대기 분리까지 한 번에 실행하는 통합 파이프라인을 완성하고 운영 문서 완비율·자체 점검·인터페이스 명세·브리핑 요약을 출력한다."""
import importlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 노선망·구간 통과량 생성기 (실제 도로·지명·부대 없음)
# 가상 출발지(SRC)에서 가상 목적지(DST)까지 3개 축선(A·B·C)과 연결로 2개, 구간 10개. 150일치 피크시간 통과량·지체 시간을 만든다.
# ---------------------------------------------------------------------------
SEGS = pd.DataFrame([
    ("A1", "SRC", "J1", 12, 130, "일반"), ("A2", "J1", "J2", 8, 70, "교량"), ("A3", "J2", "DST", 10, 140, "교차로"),
    ("B1", "SRC", "J3", 15, 100, "일반"), ("B2", "J3", "J2", 9, 80, "터널"), ("B3", "J3", "DST", 14, 95, "일반"),
    ("C1", "SRC", "J4", 18, 70, "일반"), ("C2", "J4", "DST", 20, 75, "교량"),
    ("X1", "J1", "J3", 7, 50, "교차로"), ("X2", "J4", "J3", 10, 60, "일반")],
    columns=["seg", "u", "v", "km", "cap", "kind"])               # cap = 피크시간 통과 용량(대/시, 교육용 가정값)
KSPD = {"일반": 55, "교량": 40, "터널": 45, "교차로": 45}           # 구간 유형별 자유 주행 속도(km/h, 가정값)
SEGS["t0"] = SEGS["km"] / SEGS["kind"].map(KSPD) * 60              # 자유 주행 시간(분)
UTIL = dict(A1=.78, A2=.85, A3=.66, B1=.70, B2=.72, B3=.64, C1=.55, C2=.58, X1=.45, X2=.40)   # 평시 이용률


def bpr(t0, v, cap, a=0.15, b=4.0):
    """BPR 지체 함수(교통공학의 표준형): 통과량/용량 비가 커질수록 지체가 급증한다. (분)"""
    return t0 * a * (v / cap) ** b


def make_data(seed=11, days=150):
    """구간 x 일자 패널. v=실제 피크 통과량, plan_v=수송계획 수량(긴급 불출 미반영), delay=실제 지체(분)."""
    rng = np.random.default_rng(seed)
    d_idx = np.arange(days)
    lvl = np.array([1.0, 1.08, 1.04, 1.10, 1.02, 0.82, 0.78])[d_idx % 7] * (1 + 0.0012 * d_idx)
    lvl = lvl * np.where((d_idx >= 100) & (d_idx < 110), 1.30, 1.0)       # 수송량 폭증 기간(가상)
    prev, rows = {s: 0.0 for s in SEGS["seg"]}, []
    for d in d_idx:
        rain = bool(rng.random() < 0.18)
        for r in SEGS.itertuples():
            maint, urgent = bool(rng.random() < 0.06), bool(rng.random() < 0.05)   # 비계획 정비 / 긴급 불출
            v = r.cap * UTIL[r.seg] * lvl[d] * (1.25 if urgent else 1.0) * rng.normal(1, 0.05)
            cap_eff = r.cap * (0.65 if maint else 1.0) * (0.92 if rain else 1.0)
            delay = max(r.t0 * 0.2 * (v / cap_eff) ** 3.5 + 0.25 * prev[r.seg] + rng.normal(0, 0.4), 0.0)
            prev[r.seg] = delay
            rows.append(dict(day=int(d), dow=int(d % 7), seg=r.seg, kind=r.kind, km=r.km, cap=r.cap, t0=r.t0,
                             plan_v=v / (1.25 if urgent else 1.0) * rng.normal(1, 0.04), v=v, maint=int(maint), urgent=int(urgent),
                             rain=int(rain), cap_eff=cap_eff, delay=delay))
    df = pd.DataFrame(rows)
    df["transit"] = df["t0"] * (1 + 0.08 * df["rain"]) + df["delay"]
    return df


def mape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.mean(np.abs(y - p) / np.abs(y)) * 100)


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 지체 예측용 파생 피처 생성(당일 오전 시점에 알 수 있는 정보만 사용: 계획 물량, 공지 정비, 기상 예보, 전일·전주 실적)
# ---------------------------------------------------------------------------
FEATS = ["plan_v", "cap", "cap_eff", "t0", "km", "maint", "rain", "util", "util_eff", "util_eff4", "bpr_base", "lag1_delay", "lag7_delay",
         "roll7_v", "surge_ratio", "is_weekend", "dow_sin", "dow_cos", "k_교량", "k_터널", "k_교차로"]


def add_features(df):
    d = df.sort_values(["seg", "day"]).copy()
    g = d.groupby("seg")
    d["lag1_delay"] = g["delay"].shift(1)
    d["lag7_delay"] = g["delay"].shift(7)
    d["roll7_v"] = g["v"].transform(lambda s: s.shift(1).rolling(7).mean())
    d["util"] = d["plan_v"] / d["cap"]
    d["util_eff"] = d["plan_v"] / d["cap_eff"]
    d["util_eff4"] = d["util_eff"] ** 4
    d["bpr_base"] = bpr(d["t0"], d["plan_v"], d["cap"])
    d["surge_ratio"] = d["plan_v"] / d["roll7_v"]
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["dow_sin"], d["dow_cos"] = np.sin(2 * np.pi * d["dow"] / 7), np.cos(2 * np.pi * d["dow"] / 7)
    for k in ("교량", "터널", "교차로"):
        d["k_" + k] = (d["kind"] == k).astype(int)
    return d.dropna().sort_values(["day", "seg"]).reset_index(drop=True)
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 노선망 흐름 도구: 최대유량(Edmonds-Karp), 증분 배정(혼잡 반영 최단시간 경로), 경로 열거
# ---------------------------------------------------------------------------
CAP = dict(zip(SEGS["seg"], SEGS["cap"]))
T0 = dict(zip(SEGS["seg"], SEGS["t0"]))
KM = dict(zip(SEGS["seg"], SEGS["km"]))
ARCS = {r.seg: (r.u, r.v) for r in SEGS.itertuples()}
SRC, DST = "SRC", "DST"


def seg_time(s, flow, cap=None, a=0.2, b=3.5):
    """구간 통과시간(분) = 자유주행 + 혼잡 지체. cap을 주면 용량을 대체(정비·기상 반영)."""
    r = flow / (cap or CAP[s])
    return T0[s] * (1 + a * r ** b + 6.0 * max(0.0, r - 1.0))        # 용량 초과분은 대기열로 급증


def max_flow(closed=(), capmul=None):
    """최대유량(대/시)과 최소 절단 구간(병목). capmul = {구간: 용량 배율}."""
    cap = {s: (0 if s in closed else CAP[s] * (capmul or {}).get(s, 1.0)) for s in CAP}
    res = {}
    for s, (u, v) in ARCS.items():
        res[(u, v)] = res.get((u, v), 0) + cap[s]
        res.setdefault((v, u), 0)
    flow = 0.0
    while True:
        prev, q = {SRC: None}, [SRC]
        for u in q:
            for (a, b), c in res.items():
                if a == u and c > 1e-9 and b not in prev:
                    prev[b] = u; q.append(b)
        if DST not in prev:
            break
        path, x = [], DST
        while prev[x]:
            path.append((prev[x], x)); x = prev[x]
        f = min(res[e] for e in path)
        for a, b in path:
            res[(a, b)] -= f; res[(b, a)] += f
        flow += f
    reach = set(prev)
    cut = [s for s, (u, v) in ARCS.items() if u in reach and v not in reach and cap[s] > 0]
    return flow, cut


def all_paths():
    """SRC -> DST 단순 경로(구간 이름 튜플) 전체."""
    out = []

    def dfs(node, segs, seen):
        if node == DST:
            out.append(tuple(segs)); return
        for s, (u, v) in ARCS.items():
            if u == node and v not in seen:
                dfs(v, segs + [s], seen | {v})
    dfs(SRC, [], {SRC})
    return out


PATHS = all_paths()


def assign(demand, closed=(), capmul=None, steps=40):
    """증분 배정: 수요를 steps조각으로 나눠, 전체 통과시간(대수 x 시간) 증가가 가장 작은 경로에 차례로 얹는다(시스템 최적 근사)."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    flow, pf = {s: 0.0 for s in CAP}, {}
    ok = [p for p in PATHS if not any(s in closed for s in p)]
    if not ok:
        return None
    for _ in range(steps):
        chunk = demand / steps
        best = min(ok, key=lambda p: sum((flow[s] + chunk) * seg_time(s, flow[s] + chunk, cap[s]) - flow[s] * seg_time(s, flow[s], cap[s]) for s in p))
        for s in best:
            flow[s] += chunk
        pf[best] = pf.get(best, 0) + chunk
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    avg = sum(pf[p] * tt[p] for p in pf) / demand
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": avg, "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}}


FIXED_SHARE = {("A1", "A2", "A3"): 0.32, ("B1", "B3"): 0.40, ("C1", "C2"): 0.28}      # 과거 관행: 축선별 고정 배분(가상)


def fixed_assign(demand, closed=(), capmul=None):
    """과거 관행 배분: 축선별 고정 비율. 막힌 축선 물량은 열려 있는 축선에 비율대로 재분배."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    live = {p: w for p, w in FIXED_SHARE.items() if not any(s in closed for s in p)}
    if not live:
        return None
    tot = sum(live.values())
    flow = {s: 0.0 for s in CAP}
    pf = {p: demand * w / tot for p, w in live.items()}
    for p, f in pf.items():
        for s in p:
            flow[s] += f
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": sum(pf[p] * tt[p] for p in pf) / demand,
            "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}}


def lp_assign(demand, closed=(), capmul=None):
    """선형계획(LP): 구간 용량 상한(이용률 rho) 아래에서 자유주행시간 합을 최소화하는 경로별 물량. 불가능하면 rho를 1.0까지 완화."""
    cap = {s: CAP[s] * (capmul or {}).get(s, 1.0) for s in CAP}
    ok = [p for p in PATHS if not any(s in closed for s in p)]
    if not ok:
        return None
    for rho in (0.85, 0.95, 1.0):
        r = linprog([sum(T0[s] for s in p) for p in ok], A_ub=[[1.0 if s in p else 0.0 for p in ok] for s in CAP],
                    b_ub=[rho * cap[s] for s in CAP], A_eq=[[1.0] * len(ok)], b_eq=[demand], bounds=(0, None), method="highs")
        if r.status == 0:
            break
    else:
        return assign(demand, closed, capmul)                # 용량 초과 수요: 증분 배정으로 대체(과부하 표시)
    flow = {s: sum(r.x[i] for i, p in enumerate(ok) if s in p) for s in CAP}
    pf = {p: float(r.x[i]) for i, p in enumerate(ok) if r.x[i] > 1e-6}
    tt = {p: sum(seg_time(s, flow[s], cap[s]) for s in p) for p in pf}
    return {"flow": flow, "paths": pf, "path_min": tt, "avg_min": sum(pf[p] * tt[p] for p in pf) / demand,
            "util": {s: flow[s] / cap[s] for s in cap if cap[s] > 0}, "rho": rho}


def cost_of(res, tv=1000, fuel_km=450):
    """시간당 비용(원, 교육용 가정): 대당 시간비용 tv원/분 + 유류비 fuel_km원/km."""
    return sum(f * (res["path_min"][p] * tv + sum(KM[s] for s in p) * fuel_km) for p, f in res["paths"].items())
# ---------------------------------------------------------------------------

libs = ["numpy", "pandas", "scipy", "sklearn", "matplotlib"]
env_ok = sum(importlib.util.find_spec(m) is not None for m in libs)
print(f"[0] 환경 점검: Python {sys.version_info.major}.{sys.version_info.minor}, 필수 모듈 {env_ok}/{len(libs)}")

# ---------------- 1) 데이터 갱신·정합성 검증(미션 1~2 요약) ----------------
raw = make_data(days=180)                                       # day 0~149 기존 + day 150~179 신규 입력분
d = add_features(raw)
chk = {"(구간,일자) 유일": not raw.duplicated(["seg", "day"]).any(), "결측 0": int(raw.isna().sum().sum()) == 0, "통과량 범위": bool(((raw.v >= 0) & (raw.v <= 3 * raw.cap)).all()),
       "통과시간 >= 자유주행": bool((raw.transit >= raw.t0 - 1e-9).all())}
print(f"[1] 신규 30일 포함 {len(raw)}행 정합성 검증 {sum(chk.values())}/{len(chk)} 통과")

# ---------------- 2) 자동 재학습: 드리프트 점검 -> 챌린저 학습 -> 승격 판단 ----------------
def fit(tr):
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.08, random_state=0).fit(tr[FEATS], tr["delay"])


def evaluate(m, te):
    p = te["t0"] * (1 + 0.08 * te["rain"]) + m.predict(te[FEATS])
    return mae(te["transit"], p), mape(te["transit"], p)


def retrain_pipeline(old_end, new_end, drift_mape=3.0, min_gain=0.03):
    old, new = d[d.day < old_end], d[(d.day >= old_end) & (d.day < new_end)]
    champ = fit(old)
    ch_mae, ch_mape = evaluate(champ, new)
    drift = ch_mape > drift_mape
    chall = fit(d[d.day < new_end - 10])                         # 최근 10일은 검증용으로 남김
    hold = d[(d.day >= new_end - 10) & (d.day < new_end)]
    a, b = evaluate(champ, hold)[0], evaluate(chall, hold)[0]
    promote = (a - b) / a >= min_gain
    return {"champ_mape": ch_mape, "drift": drift, "champ_mae": a, "chall_mae": b, "promote": promote, "model": chall if promote else champ}


rt = retrain_pipeline(150, 180)
print(f"[2] 재학습 파이프라인: 기존 모델의 신규 30일 MAPE {rt['champ_mape']:.2f}% (드리프트 기준 3.0%: {'초과' if rt['drift'] else '이내'})")
print(f"    검증 10일 MAE 챔피언 {rt['champ_mae']:.3f}분 vs 챌린저 {rt['chall_mae']:.3f}분 -> {'챌린저 승격' if rt['promote'] else '기존 모델 유지'}(개선 3% 이상일 때만 승격)")
model = rt["model"]

# ---------------- 3) 7일 수송 소요 예측(Ridge 요일+추세 vs 전주 동요일 Baseline) ----------------
daily = raw[raw.seg.isin(["A1", "B1", "C1"])].groupby("day")["plan_v"].sum()


def design(days):
    days = np.asarray(days)
    X = np.zeros((len(days), 9))
    X[np.arange(len(days)), days % 7] = 1
    X[:, 7] = days / 100
    X[:, 8] = ((days >= 100) & (days < 110)).astype(float)        # 폭증 기간(알려진 사건) 더미
    return X


tr_days = daily.index[daily.index < 173]
rg = Ridge(alpha=0.1, fit_intercept=False).fit(design(tr_days), daily[tr_days])
test_days = np.arange(173, 180)
f_ridge = rg.predict(design(test_days)); f_naive = daily[test_days - 7].values
print(f"\n[3] 수송 소요 7일 예측 검증(day 173~179): Ridge MAPE {mape(daily[test_days], f_ridge):.1f}% vs 전주 동요일 {mape(daily[test_days], f_naive):.1f}%")
rg = Ridge(alpha=0.1, fit_intercept=False).fit(design(daily.index), daily)
nxt = np.arange(180, 187)
dem = rg.predict(design(nxt))

# ---------------- 4) 소요 -> 병목·우회 배분 -> 승인대기 분리 ----------------
NOTICE = {183: {"A2": 0.65}}                                    # 공지된 정비(가정): day 183 A2 교량 용량 65%
out = []
for day, D in zip(nxt, dem):
    cm = NOTICE.get(int(day))
    f, l = fixed_assign(D, (), cm), lp_assign(D, (), cm)
    extra = l["avg_min"] - lp_assign(D)["avg_min"]
    risky = max(l["util"].values()) > 0.98 or extra >= 3 or day in NOTICE
    out.append({"day": int(day), "예측 소요(대/시)": D, "고정(분)": f["avg_min"], "우회 배정(분)": l["avg_min"], "절감률": 1 - cost_of(l) / cost_of(f),
                "최대 이용률": max(l["util"].values()), "처리": "작전군수관 승인대기(2단계)" if risky else "추천(검토 후 집행)"})
pl = pd.DataFrame(out)
print("\n[4] 향후 7일 수송 소요·우회 배분 계획 [교육용가상자료]")
print(pl.round(2).to_string(index=False))
pend = pl[pl["처리"].str.contains("승인대기")]
print(f"    승인대기 {len(pend)}건(day {', '.join(map(str, pend['day']))}) - 자동 집행하지 않고 작전군수관 결재(미션 7)로 넘긴다.")

# ---------------- 5) 운영 문서 완비율·인터페이스 명세 ----------------
docs = {"데이터 사전·스키마(M1)": 29, "정제 규칙 10종(M2)": 10, "Baseline·EDA(M3)": 8, "병목 식별·우회(M4)": 4, "우회 소요·유류 엔진(M5)": 4, "대시보드·설명(M6)": 6,
        "승인 절차·권한(M7)": 4, "스트레스 시험(M8)": 4, "AAR·ROI(M9)": 5, "운영·재학습 절차(M10)": 6, "장애 대응": 5, "인터페이스 가이드": 4}
done = sum(v > 0 for v in docs.values())
iface = {"input": {"daily_plan": ["day:int", "seg:str", "plan_v:float", "maint:int", "rain:int"], "format": "json-lines"},
         "output": {"forecast": ["day", "demand_vph", "paths", "avg_min", "status"], "approval": "PENDING_ONLY"}, "auth": "role-based(가상)", "note": "상위 군수정보체계는 가상 인터페이스로만 정의"}
print(f"\n[5] 운영 문서 {len(docs)}개 항목 중 작성 완료 {done}개 ({done / len(docs):.0%}) / 인터페이스 명세(JSON):")
print("   ", json.dumps(iface, ensure_ascii=False))

# ---------------- 6) 자체 점검·브리핑 ----------------
te = d[d.day >= 150]
fin_mae, fin_mape = evaluate(model, te)
base_mape = mape(te.transit, te.t0 + te.bpr_base)
checks = [("환경 점검 통과", env_ok == len(libs)), ("데이터 정합성 전 항목 통과", all(chk.values())), ("재학습 파이프라인 정상 실행(챔피언/챌린저 판정 출력)", rt["champ_mae"] > 0),
          ("통과시간 예측 MAPE 25% 이하", fin_mape <= 25), ("기준 BPR 대비 MAE 20% 이상 개선", fin_mae <= 0.8 * mae(te.transit, te.t0 + te.bpr_base)),
          ("7일 소요 예측이 전주 동요일 Baseline보다 정확", mape(daily[test_days], f_ridge) < mape(daily[test_days], f_naive)),
          ("위험·정비 우회안 자동 집행 0건(승인대기 분리)", len(pend) >= 1 and (pl.loc[pl.day == 183, "처리"].str.contains("승인대기")).all()), ("운영 문서 100% 완비", done == len(docs))]
print("\n[6] 자체 점검")
for n, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {n}")
print("\n[7] 브리핑 요약")
print(f"  - 신규 30일 데이터로 재학습 판단을 자동 수행했다(드리프트 {'감지' if rt['drift'] else '미감지'}, {'챌린저 승격' if rt['promote'] else '기존 모델 유지'}).")
print(f"  - 통과시간 MAPE {base_mape:.1f}%(BPR) -> {fin_mape:.1f}%, 향후 7일 소요 예측 MAPE {mape(daily[test_days], f_ridge):.1f}%.")
print(f"  - 향후 7일 중 우회 최적화 평균 절감 {pl['절감률'].mean():.1%}, 정비 공지일은 승인대기로 분리했다. 실제 우회 하달은 작전군수관 결재와 지휘계통 결심에 따른다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(daily.index, daily.values, label="actual"); ax[0].plot(nxt, dem, "r--o", label="forecast"); ax[0].legend(); ax[0].set_title("Daily transport demand and 7-day forecast")
ax[1].bar(pl["day"] - .2, pl["고정(분)"], .4, label="fixed"); ax[1].bar(pl["day"] + .2, pl["우회 배정(분)"], .4, label="detour assignment"); ax[1].legend(); ax[1].set_title("Avg transit (min), next 7 days")
plt.suptitle("MOV01 integrated pipeline (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[0] 환경 점검: Python 3.11, 필수 모듈 5/5
[1] 신규 30일 포함 1800행 정합성 검증 4/4 통과
[2] 재학습 파이프라인: 기존 모델의 신규 30일 MAPE 2.93% (드리프트 기준 3.0%: 이내)
    검증 10일 MAE 챔피언 0.534분 vs 챌린저 0.535분 -> 기존 모델 유지(개선 3% 이상일 때만 승격)

[3] 수송 소요 7일 예측 검증(day 173~179): Ridge MAPE 3.5% vs 전주 동요일 4.7%

[4] 향후 7일 수송 소요·우회 배분 계획 [교육용가상자료]
 day  예측 소요(대/시)  고정(분)  우회 배정(분)  절감률  최대 이용률              처리
 180      216.46  42.89     42.43 0.00    0.85     추천(검토 후 집행)
 181      204.40  42.15     41.95 0.00    0.85     추천(검토 후 집행)
 182      258.65  56.35     44.70 0.16    0.95     추천(검토 후 집행)
 183      276.73  91.80     51.32 0.37    1.11 작전군수관 승인대기(2단계)
 184      266.37  61.40     45.29 0.21    0.95     추천(검토 후 집행)
 185      275.57  68.96     45.87 0.27    1.00 작전군수관 승인대기(2단계)
 186      259.89  57.16     44.79 0.17    0.95     추천(검토 후 집행)
    승인대기 2건(day 183, 185) - 자동 집행하지 않고 작전군수관 결재(미션 7)로 넘긴다.

[5] 운영 문서 12개 항목 중 작성 완료 12개 (100%) / 인터페이스 명세(JSON):
    {"input": {"daily_plan": ["day:int", "seg:str", "plan_v:float", "maint:int", "rain:int"], "format": "json-lines"}, "output": {"forecast": ["day", "demand_vph", "paths", "avg_min", "status"], "approval": "PENDING_ONLY"}, "auth": "role-based(가상)", "note": "상위 군수정보체계는 가상 인터페이스로만 정의"}

[6] 자체 점검
  [PASS] 환경 점검 통과
  [PASS] 데이터 정합성 전 항목 통과
  [PASS] 재학습 파이프라인 정상 실행(챔피언/챌린저 판정 출력)
  [PASS] 통과시간 예측 MAPE 25% 이하
  [PASS] 기준 BPR 대비 MAE 20% 이상 개선
  [PASS] 7일 소요 예측이 전주 동요일 Baseline보다 정확
  [PASS] 위험·정비 우회안 자동 집행 0건(승인대기 분리)
  [PASS] 운영 문서 100% 완비

[7] 브리핑 요약
  - 신규 30일 데이터로 재학습 판단을 자동 수행했다(드리프트 미감지, 기존 모델 유지).
  - 통과시간 MAPE 9.6%(BPR) -> 2.9%, 향후 7일 소요 예측 MAPE 3.5%.
  - 향후 7일 중 우회 최적화 평균 절감 16.9%, 정비 공지일은 승인대기로 분리했다. 실제 우회 하달은 작전군수관 결재와 지휘계통 결심에 따른다.
```
</details>
